In [2]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, asin, sqrt
import json
from collections import defaultdict

In [3]:
pd.set_option('display.max_columns', None)

## Einlesen der Daten

In [57]:
soll = pd.read_csv('transport_optimization/Soll-Daten.csv', dtype=str)
soll.head()

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,PLZ,ORT,STRASSE,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,20457.0,Hamburg,Brandenburger Straße 12,9.98632,53.52348,0 days 00:30:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,20457.0,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,20457.0,Hamburg,Brandenburger Str. 12,9.98632,53.52348,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,20457.0,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,0 days 00:30:00,2026-03-02
4,H/42375,OD-TS8050,"Szmajdewicz, Marcin",H/42375,2026-03-02 06:00:00,2026-03-02 06:30:00,20457.0,Hamburg,Brandenburger Str. 12,9.98632,53.52348,0 days 00:30:00,2026-03-02


In [42]:
ist = pd.read_csv('transport_optimization/Ist-Daten.csv', dtype=str)
ist.head()

,Fahrer PIN,Fahrername,Formularnummer,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR,Abfahrt,Ankunft
0,8160.0,"Jorczik, Christian",1650,0,0,317003.22,0.0,0.0,"53,6677","9,99287","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 06:17:22,2026-03-30 05:18:19
1,8160.0,"Jorczik, Christian",1650,131,0,317021.295,0.0,0.0,"53,55026","9,99017","DE - 20457 Hamburg (Hamburg-Mitte), Alter Wall 32",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 06:59:55,2026-03-30 06:59:54
2,8160.0,"Jorczik, Christian",1650,0,0,317028.945,0.0,0.0,"53,57519","9,9127","DE - 22525 Hamburg (Bahrenfeld), Winsbergring 15",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 07:28:24,2026-03-30 07:28:23
3,8160.0,"Jorczik, Christian",1650,0,0,317066.35,0.0,0.0,"53,61828","10,23092","DE - 22145 Braak, Waldweg 2",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 08:37:25,2026-03-30 08:37:24
4,8160.0,"Jorczik, Christian",1650,0,0,317080.55,0.0,0.0,"53,67164","10,23997","DE - 22926 Ahrensburg, Manhagener Allee 10",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 09:24:07,2026-03-30 09:24:06


## Touren mit gleicher Reihenfolge an Stopps filtern
### Hilfsfunktionen für Geofencing

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Berechnet den kürzesten Abstand zwischen zwei Punkten auf einer Kugeloberfläche."""
    # Erdradius in Metern
    R = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

def parse_decimal_coord(x):
    """Konvertiert eine Koordinate, die als String mit möglichen Tausendertrennzeichen und Komma als Dezimaltrennzeichen vorliegen kann, in einen float-Wert."""
    # komma zu punkt in dezimalzahl
    if pd.isna(x):
        return np.nan
    s = str(x).replace(' ', '').replace('"','')
    s = s.replace(',', '.') if (',' in s and '.' not in s) else s
    return float(s)

MATCH_RADIUS_M = 300  # max Entfernung zwischen Punkten, damit sie als Match gelten (Geofencing-Radius)

### Vorverarbeitung Soll-Daten

In [44]:
# Erwartete Spalten: TOURNR,GEOX,GEOY,ANKUNFT
soll['GEOX'] = soll['GEOX'].astype(str).apply(parse_decimal_coord)
soll['GEOY'] = soll['GEOY'].astype(str).apply(parse_decimal_coord)
soll['ANKUNFT'] = pd.to_datetime(soll['ANKUNFT'], errors='coerce')
soll = soll.sort_values(['TOURNR','ANKUNFT']).reset_index(drop=True)

# Erzeuge für jede TOURNR eine Liste von (lat,lon)
soll_groups = {}
for tournr, g in soll.groupby('TOURNR'):
    coords = list(zip(g['GEOY'].astype(float).tolist(), g['GEOX'].astype(float).tolist()))
    soll_groups[tournr] = coords

### Vorverarbeitung Ist-Daten

In [ ]:
# Erwartete Spalten: TOURNR,Breitengrad,Längengrad,Ankunft (oder Abfahrt)
# Normiere Spaltennamen (in Dateien sind viele Varianten); wir nutzen 'TOURNR','Breitengrad','Längengrad','Ankunft'
ist_cols = {c.lower(): c for c in ist.columns}
# Mögliche Spaltennamen: 'Breitengrad' oder 'BREITENGRAD' etc. Wir finden passende.
def find_col(names):
    for n in names:
        if n.lower() in ist_cols:
            return ist_cols[n.lower()]
    return None

col_lat = find_col(['Breitengrad','breitengrad','BREITENGRAD','LAT','LATITUDE','B_LAT'])
col_lon = find_col(['Längengrad','Längengrad','LÄNGENGRAD','LON','LONGITUDE','B_LON'])
col_time = find_col(['Ankunft','ankunft','ANREISE','Ankunftszeit','Abfahrt','abfahrt'])
col_tournr = find_col(['TOURNR','tournr','Tournummer','TOUR','TOURNR'])

if col_lat is None or col_lon is None or col_tournr is None:
    raise ValueError("Konnte benötigte Spalten in Ist-Daten nicht finden. Gefundene Spalten: " + ", ".join(ist.columns))

ist['lat'] = ist[col_lat].apply(parse_decimal_coord)
ist['lon'] = ist[col_lon].apply(parse_decimal_coord)
ist[col_time] = pd.to_datetime(ist[col_time], errors='coerce')
ist = ist.sort_values([col_tournr, col_time]).reset_index(drop=True)

# Gruppiere Ist pro TOURNR
ist_groups = {}
for tournr, g in ist.groupby(col_tournr):
    coords = list(zip(g['lat'].astype(float).tolist(), g['lon'].astype(float).tolist()))
    ist_groups[tournr] = coords

### Finde Touren, die matchen

In [ ]:
def tour_sequence_matches_strict(
    istseq,
    sollseq,
    radiusm=MATCH_RADIUS_M,
    min_match_ratio=0.8,
    require_unique_soll=False,
    allow_skips=True,
):
    """
    Prüft, ob Ist dieselbe Stop-Reihenfolge wie Soll abfährt.

    Regeln:
    - Pro Ist-Punkt wird der nächste Soll-Punkt innerhalb radius gesucht.
    - Die Folge der gematchten Soll-Indizes muss streng aufsteigend sein
      (keine Wiederholung desselben Soll-Stopps).
    - Optional kann gefordert werden, dass jeder Soll-Stopp nur einmal gematcht wird.
    - Mindestanteil gematchter Soll-Stopps über min_match_ratio.

    Returns
    -------
    matches : bool
    details : dict
    """
    matched = []

    for i, (ilat, ilon) in enumerate(istseq):
        best_idx = None
        best_dist = float("inf")

        for j, (slat, slon) in enumerate(sollseq):
            d = haversine(ilat, ilon, slat, slon)
            if d < best_dist:
                best_dist = d
                best_idx = j

        if best_dist <= radiusm:
            matched.append({
                "ist_idx": i,
                "soll_idx": best_idx,
                "dist_m": best_dist
            })
        else:
            matched.append({
                "ist_idx": i,
                "soll_idx": None,
                "dist_m": None
            })

    valid = [m for m in matched if m["soll_idx"] is not None]
    matched_indices = [m["soll_idx"] for m in valid]

    if len(valid) == 0:
        return False, {
            "matched": matched,
            "matched_count": 0,
            "soll_count": len(sollseq),
            "ist_count": len(istseq),
            "coverage_ratio": 0.0,
            "strictly_increasing": False,
            "unique_soll_count": 0,
            "matched_indices": [],
            "reason": "kein Punkt innerhalb Radius"
        }

    strictly_increasing = all(
        a < b for a, b in zip(matched_indices, matched_indices[1:])
    )

    unique_soll_count = len(set(matched_indices))
    coverage_ratio = unique_soll_count / len(sollseq) if len(sollseq) else 0.0

    if require_unique_soll:
        unique_ok = len(matched_indices) == unique_soll_count
    else:
        unique_ok = True

    if allow_skips:
        coverage_ok = coverage_ratio >= min_match_ratio
    else:
        coverage_ok = unique_soll_count == len(sollseq)

    matches = strictly_increasing and unique_ok and coverage_ok

    return matches, {
        "matched": matched,
        "matched_count": len(valid),
        "soll_count": len(sollseq),
        "ist_count": len(istseq),
        "coverage_ratio": coverage_ratio,
        "strictly_increasing": strictly_increasing,
        "unique_soll_count": unique_soll_count,
        "matched_indices": matched_indices,
        "reason": None if matches else {
            "strictly_increasing": strictly_increasing,
            "unique_ok": unique_ok,
            "coverage_ok": coverage_ok
        }
    }


# Prüfe alle TOURNR, die in beiden Sets vorkommen 
commontournr = set(soll_groups.keys()).intersection(set(ist_groups.keys()))
matchingtournr = []
matchdetails = {}

for t in sorted(commontournr):
    istseq = ist_groups[t]
    sollseq = soll_groups[t]

    matches, details = tour_sequence_matches_strict(
        istseq,
        sollseq,
        radiusm=MATCH_RADIUS_M,
        min_match_ratio=0.8,
        require_unique_soll=False,
        allow_skips=True
    )

    matchdetails[t] = details

    if matches:
        matchingtournr.append(t)


# Join der DataFrames für gematchte TOURNR 
matchedsoll = soll[soll["TOURNR"].isin(matchingtournr)].copy()
matchedist = ist[ist[col_tournr].isin(matchingtournr)].copy()

# Optional: nummeriere Stop-Positionen innerhalb jeder Tour
matchedsoll["sollstopidx"] = matchedsoll.groupby("TOURNR").cumcount() + 1
matchedist["iststopidx"] = matchedist.groupby(col_tournr).cumcount() + 1

# Einfacher Join auf TOURNR
merged = pd.merge(
    matchedist,
    matchedsoll,
    left_on=col_tournr,
    right_on="TOURNR",
    suffixes=("_ist", "_soll")
)

# Speichere Ergebnisse
pd.DataFrame({"matchingtournr": list(matchingtournr)}).to_csv("matchingtournr.csv", index=False)
matchedsoll.to_csv("matchedsoll.csv", index=False)
matchedist.to_csv("matchedist.csv", index=False)
merged.to_csv("mergedmatchedtours.csv", index=False)

# Speichere Detail-Report
with open("matchdetails.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "MATCH_RADIUS_M": MATCH_RADIUS_M,
            "matchingtournr": matchingtournr,
            "details": matchdetails
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Fertig. Gefundene gematchte Touren: {len(matchingtournr)}. "
    "Listen in matchingtournr.csv, mergedmatchedtours.csv gespeichert."
)

Fertig. Gefundene gematchte Touren: 14. Listen in matchingtournr.csv, mergedmatchedtours.csv gespeichert.


### Sauberer Pair-Merge auf Basis der echten Matches
* bauen aus `matchdetails[t]["matched"]` eine echte Match-Tabelle und nehmen dann die laufenden Indizes ist_idx und soll_idx je Tour und merged genau auf diese Paarung statt nur auf TOURNR
* daraus dann pro Match eine Zeile mit Distanz, Ist-Zeit, Soll-Zeit und daraus delay_min
* Matching-Logik kann mehrere Ist-Punkte demselben Soll-Stopp zuordnen, weil pro Ist-Punkt der nächste Soll-Punkt gesucht wird -> paired_best reduziert das auf genau einen besten Kandidaten pro TOURNR + soll_idx, nämlich den mit der kleinsten Distanz, was für Stop-basierte Delay-Auswertungen meist die sauberste Variante ist

In [ ]:
pair_rows = []

for tournr in matchingtournr:
    details = matchdetails[tournr]

    for m in details["matched"]:
        if m["soll_idx"] is None:
            continue

        pair_rows.append({
            "TOURNR": tournr,
            "ist_idx": m["ist_idx"],
            "soll_idx": m["soll_idx"],
            "dist_m": m["dist_m"]
        })

pairs = pd.DataFrame(pair_rows)

# Falls keine Paare gefunden wurden
if pairs.empty:
    raise ValueError("Keine gültigen Match-Paare in matchdetails gefunden.")

# Eindeutige laufende Indizes je Tour in Originaldaten

matchedist = ist[ist[col_tournr].isin(matchingtournr)].copy()
matchedsoll = soll[soll["TOURNR"].isin(matchingtournr)].copy()

matchedist = matchedist.sort_values([col_tournr, col_time]).reset_index(drop=True)
matchedsoll = matchedsoll.sort_values(["TOURNR", "ANKUNFT"]).reset_index(drop=True)

matchedist["ist_idx"] = matchedist.groupby(col_tournr).cumcount()
matchedsoll["soll_idx"] = matchedsoll.groupby("TOURNR").cumcount()

# Nur benötigte Spalten für Ist / Soll vorbereiten

ist_cols_keep = [col_tournr, "ist_idx", col_time, "lat", "lon"]
for c in ["Abfahrt", "Ankunft", "Position", "Fahrzeug", "KENNZCLEAN", "DATUM"]:
    if c in matchedist.columns and c not in ist_cols_keep:
        ist_cols_keep.append(c)

soll_cols_keep = ["TOURNR", "soll_idx", "ANKUNFT", "ABFAHRT", "GEOY", "GEOX"]
for c in ["STRASSE", "ORT", "PLZ", "LKWKENNZ", "FAHRERNAME", "DATUM"]:
    if c in matchedsoll.columns and c not in soll_cols_keep:
        soll_cols_keep.append(c)

ist_for_merge = matchedist[ist_cols_keep].copy()
soll_for_merge = matchedsoll[soll_cols_keep].copy()

# Umbenennen für Klarheit
ist_for_merge = ist_for_merge.rename(columns={
    col_tournr: "TOURNR",
    col_time: "IstZeit",
    "Ankunft": "IstAnkunft",
    "Abfahrt": "IstAbfahrt",
    "Position": "IstPosition",
    "Fahrzeug": "IstFahrzeug",
    "DATUM": "DATUMist"
})

soll_for_merge = soll_for_merge.rename(columns={
    "ANKUNFT": "SollAnkunft",
    "ABFAHRT": "SollAbfahrt",
    "GEOY": "SollLat",
    "GEOX": "SollLon",
    "STRASSE": "SollStrasse",
    "ORT": "SollOrt",
    "PLZ": "SollPLZ",
    "DATUM": "DATUMsoll"
})

# Echter Merge über TOURNR + Indexpaar

paired = pairs.merge(
    ist_for_merge,
    on=["TOURNR", "ist_idx"],
    how="left"
).merge(
    soll_for_merge,
    on=["TOURNR", "soll_idx"],
    how="left"
)

# Delay berechnen
# Falls IstZeit nicht brauchbar ist, nimm bevorzugt IstAnkunft, sonst IstAbfahrt

if "IstAnkunft" in paired.columns:
    paired["IstZeitFinal"] = pd.to_datetime(paired["IstAnkunft"], errors="coerce")
else:
    paired["IstZeitFinal"] = pd.NaT

if "IstAbfahrt" in paired.columns:
    ist_abfahrt_dt = pd.to_datetime(paired["IstAbfahrt"], errors="coerce")
    paired["IstZeitFinal"] = paired["IstZeitFinal"].fillna(ist_abfahrt_dt)

paired["SollZeitFinal"] = pd.to_datetime(paired["SollAnkunft"], errors="coerce")

paired["delay_min"] = (
    (paired["IstZeitFinal"] - paired["SollZeitFinal"]).dt.total_seconds() / 60.0
)

# nur beste/saubere Paare behalten
# Falls mehrere Ist-Punkte auf denselben Soll-Stopp zeigen,
# behalte den mit der kleinsten Distanz

paired_best = (
    paired.sort_values(["TOURNR", "soll_idx", "dist_m"])
          .drop_duplicates(subset=["TOURNR", "soll_idx"], keep="first")
          .sort_values(["TOURNR", "soll_idx"])
          .reset_index(drop=True)
)

# Exporte

pairs.to_csv("match_pairs_raw.csv", index=False)
paired.to_csv("matched_pairs_full.csv", index=False)
paired_best.to_csv("matched_pairs_best.csv", index=False)

# Optional: Delay-Zusammenfassung pro Tour
tour_delay_summary = (
    paired_best.groupby("TOURNR", dropna=False)
    .agg(
        n_pairs=("soll_idx", "count"),
        avg_delay_min=("delay_min", "mean"),
        median_delay_min=("delay_min", "median"),
        max_delay_min=("delay_min", "max"),
        min_delay_min=("delay_min", "min")
    )
    .reset_index()
)

tour_delay_summary.to_csv("tour_delay_summary.csv", index=False)

print(
    f"Fertig. {len(pairs)} Roh-Paare, "
    f"{len(paired_best)} eindeutige beste Paare gespeichert."
)

Fertig. 58 Roh-Paare, 58 eindeutige beste Paare gespeichert.


In [52]:
paired_best.TOURNR.nunique()

14

In [55]:
paired_best[paired_best.TOURNR == 'H/42675']

,TOURNR,ist_idx,soll_idx,dist_m,IstAnkunft,lat,lon,IstAbfahrt,IstPosition,IstFahrzeug,DATUMist,SollAnkunft,SollAbfahrt,SollLat,SollLon,SollStrasse,SollOrt,SollPLZ,FAHRERNAME,DATUMsoll,IstZeitFinal,SollZeitFinal,delay_min
50,H/42675,0,0,40.905876,2026-04-02 05:20:51,53.52339,9.98572,2026-04-02 05:20:59,"DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-04-02,2026-04-02 06:00:00,2026-04-02 06:30:00,53.52348,9.98632,Brandenburger Straße 12,Hamburg,20457.0,"Hesse, Sven-Erik",2026-04-02,2026-04-02 05:20:51,2026-04-02 06:00:00,-39.15
51,H/42675,1,1,32.253817,NaT,51.93181,10.32280,2026-04-02 11:06:00,"DE - 38685 Langelsheim, Innerstetal 2",Plö-TS856 - Fahrzeugdetails,2026-04-02,2026-04-02 09:35:00,2026-04-02 10:05:00,51.93210,10.32281,Innerstetal 2,Langelsheim,38685.0,"Hesse, Sven-Erik",2026-04-02,2026-04-02 11:06:00,2026-04-02 09:35:00,91.00


In [53]:
soll.TOURNR.nunique()

240

In [54]:
ist.TOURNR.nunique()

200

In [60]:
soll[soll["TOURNR"] == 'H/42446']

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,PLZ,ORT,STRASSE,GEOX,GEOY,STOPP_DAUER,DATUM
317,H/42446,Plo-TS856,"Hesse, Sven-Erik",H/42446,2026-03-09 06:00:00,2026-03-09 06:30:00,20457.0,Hamburg,Brandenburger Straße 12,9.98632,53.52348,0 days 00:30:00,2026-03-09
318,H/42446,Plo-TS856,"Hesse, Sven-Erik",H/42446,2026-03-09 08:57:00,2026-03-09 09:27:00,26180.0,Rastede,Hohe Looge 2-8,8.17018,53.27581,0 days 00:30:00,2026-03-09


In [61]:
ist[ist[col_tournr] == 'H/42446']

,Fahrer PIN,Fahrername,Formularnummer,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR,Abfahrt,Ankunft,lat,lon
256,8317.0,"Hesse, Sven-Erik",1650,0,0,340049.615,0.0,0.0,"53,52288","9,98846","DE - 20457 Hamburg (Hamburg-Mitte), Brandenbur...",Plö-TS856 - Fahrzeugdetails,2026-03-09,Plo-TS856,H/42446,2026-03-09 05:39:02,2026-03-09 05:38:59,53.52288,9.98846
257,8317.0,"Hesse, Sven-Erik",1651,0,0,340226.07,0.0,0.0,"53,27393","8,17244","DE - 26180 Rastede, Hohe Looge 8",Plö-TS856 - Fahrzeugdetails,2026-03-09,Plo-TS856,H/42446,2026-03-09 09:52:45,NaT,53.27393,8.17244
